# 말로 장부 작성하기 — 받아적기 서버 (코랩 GPU)

**이 노트북이 하는 일은 받아적기(Whisper) 하나뿐이다.**

소리를 받아 글자로 돌려준다. 그 글자를 장부 칸으로 나누는 일(젬마)은
**내 노트북에 이미 깔려 있는 Ollama** 가 한다 — 브라우저가 직접 부른다.
여기서 젬마를 또 깔지 않는다.

```
🎤 브라우저 ──소리──▶ 코랩 (이 노트북, Whisper, GPU)
     ▲                      │
     └──────글자────────────┘
     │
     └──글자──▶ 내 노트북의 Ollama (gemma3:4b) ──장부 칸──▶ 브라우저
```

- 런타임 → 런타임 유형 변경 → **T4 GPU**
- GPU 가 없으면 CPU 로 그대로 돈다. 느릴 뿐이다
- **위에서부터 모두 실행.** 맨 아래 칸이 주소 하나를 찍어준다


## 1. 설치

In [ ]:
!pip -q install openai-whisper fastapi "uvicorn[standard]" python-multipart nest_asyncio
print("설치 끝")

## 2. Whisper 올리기 — GPU 없으면 CPU 로 간다

In [ ]:
import torch, whisper

장치 = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", 장치)
if 장치 == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 가 안 잡혔습니다. CPU 로 돕니다 (느립니다).")

소리모델 = whisper.load_model("small", device=장치)
print("Whisper small 올라감")

## 3. 서버 켜기

소리를 받아 글자로 돌려준다. 그것만 한다.

In [ ]:
import tempfile, os, threading, time, nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

앱 = FastAPI()

# 사이트가 어디에 떠 있든 부를 수 있게 열어둔다.
# 노트북을 닫으면 사라지는 시연용 서버다.
앱.add_middleware(CORSMiddleware, allow_origins=["*"],
                  allow_methods=["*"], allow_headers=["*"])

@앱.get("/health")
def 살았니():
    return {"ok": True, "장치": 장치}

@앱.post("/stt")
async def 받아적기(audio: UploadFile = File(...)):
    끝 = os.path.splitext(audio.filename or "")[1] or ".webm"
    with tempfile.NamedTemporaryFile(suffix=끝, delete=False) as 임시:
        임시.write(await audio.read())
        소리파일 = 임시.name

    시작 = time.time()
    try:
        말 = 소리모델.transcribe(소리파일, language="ko",
                               fp16=(장치 == "cuda"))["text"].strip()
    finally:
        os.unlink(소리파일)

    return {"텍스트": 말, "초": round(time.time() - 시작, 2), "장치": 장치}

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(앱, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()
time.sleep(3)
print("서버 떴다 — 8000번")

## 4. 바깥에서 들어올 주소 만들기

코랩은 바깥에서 바로 못 부른다. 터널을 뚫어 `https://...trycloudflare.com` 주소를 받는다.
**가입도 키도 필요 없다.** 노트북을 닫으면 주소는 사라진다 — 다시 켜면 주소가 바뀐다.

In [ ]:
import subprocess, re, time

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

기록 = open("/content/tunnel.log", "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000",
                  "--no-autoupdate"], stdout=기록, stderr=기록)

주소 = ""
for _ in range(60):
    time.sleep(1)
    글 = open("/content/tunnel.log", errors="ignore").read()
    찾음 = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", 글)
    if 찾음:
        주소 = 찾음.group(0)
        break

if 주소:
    print()
    print("=" * 62)
    print("  사이트의 [받아적기 주소] 칸에 이것을 붙여넣으세요")
    print()
    print("     " + 주소)
    print()
    print("=" * 62)
else:
    print("주소를 못 받았습니다. 이 칸을 다시 실행해보세요.")
    print(open("/content/tunnel.log", errors="ignore").read()[-2000:])

---

## 내 노트북 쪽에서 할 일

젬마는 **내 노트북의 Ollama** 가 돌린다. 코랩과 상관없다.

윈도우 파워셸에서 확인만 하면 된다:

```powershell
ollama list
```

`gemma3:4b` 가 보이면 끝이다. 브라우저가 `http://127.0.0.1:11434` 로 직접 부른다.

> 브라우저에서 CORS 오류가 나면 파워셸에서 한 번만:
> ```powershell
> setx OLLAMA_ORIGINS "*"
> ```
> 그리고 Ollama 를 껐다 켠다.

## 서버가 사는 동안 이 노트북을 닫지 마세요

다음에 쓸 때는 **1번부터 다시** 실행하고, 새로 나온 주소를 사이트에 다시 넣는다.